In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')
dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Donovan Mitchell,Over,27.5,-137,2025-11-14,2025-11-13T18:04:13Z
1,PrizePicks,player_points,Donovan Mitchell,Under,27.5,-137,2025-11-14,2025-11-13T18:04:13Z
2,PrizePicks,player_points,Brandon Ingram,Over,20.0,-137,2025-11-14,2025-11-13T18:04:13Z
3,PrizePicks,player_points,Brandon Ingram,Under,20.0,-137,2025-11-14,2025-11-13T18:04:13Z
4,PrizePicks,player_points,Evan Mobley,Over,19.5,-137,2025-11-14,2025-11-13T18:04:13Z


### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Applications/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 6 teams with confirmed lineups


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=10, 
                             variance_inflation=1.1, 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(10)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 43 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Pascal Siakam,Bovada,21.5,11.67,Under,210,1,20.59,205.9,0.980,Low
1,Devin Booker,Bovada,24.5,12.85,Under,200,1,19.90,199.0,0.995,Low
2,Lauri Markkanen,Bovada,21.5,11.79,Under,190,1,18.70,187.0,0.984,Low
3,Donovan Mitchell,Bovada,23.5,14.14,Under,185,1,18.28,182.8,0.988,Low
4,Pascal Siakam,Bovada,22.5,11.67,Under,170,1,16.76,167.6,0.986,Low


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 29 players...
Processing 27 players with valid predictions...
Generated 236 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Lauri Markkanen,Pascal Siakam,25.5,25.5,under,under,1,16.95,0.848,Low,Low
1,Jamal Shead,Lauri Markkanen,5.5,25.5,over,under,1,16.39,0.819,Med,Low
2,Jamal Shead,Pascal Siakam,5.5,25.5,over,under,1,16.34,0.817,Med,Low
3,Lauri Markkanen,Oso Ighodaro,25.5,5.5,under,over,1,16.06,0.803,Low,Med
4,Gradey Dick,Pascal Siakam,7.5,25.5,over,under,1,16.05,0.803,Low,Low


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1,
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 43 players...
Processing 41 players with valid predictions...
Generated 560 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Donovan Mitchell,Devin Booker,27.5,28.5,under,under,1,16.99,0.850,Low,Low
1,Donovan Mitchell,Lauri Markkanen,27.5,25.5,under,under,1,16.98,0.849,Low,Low
2,Lauri Markkanen,Devin Booker,25.5,28.5,under,under,1,16.98,0.849,Low,Low
3,Lauri Markkanen,Pascal Siakam,25.5,25.5,under,under,1,16.96,0.848,Low,Low
4,Donovan Mitchell,Pascal Siakam,27.5,25.5,under,under,1,16.96,0.848,Low,Low


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
                     variance_inflation=1.1, 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 29 players...
Processing 27 players with valid predictions...
Generated 2614 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Jamal Shead,Lauri Markkanen,Pascal Siakam,5.5,25.5,25.5,over,under,under,1,37.57,0.751,Med,Low,Low
1,Lauri Markkanen,Pascal Siakam,Oso Ighodaro,25.5,25.5,5.5,under,under,over,1,36.93,0.739,Low,Low,Med
2,Gradey Dick,Lauri Markkanen,Pascal Siakam,7.5,25.5,25.5,over,under,under,1,36.91,0.738,Low,Low,Low
3,Gradey Dick,Jamal Shead,Lauri Markkanen,7.5,5.5,25.5,over,over,under,1,36.07,0.721,Low,Med,Low
4,Jamal Shead,Lauri Markkanen,Oso Ighodaro,5.5,25.5,5.5,over,under,over,1,36.04,0.721,Med,Low,Med


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=100, 
                     variance_inflation=1.1, 
                     use_monte_carlo=False, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 43 players...
Processing 41 players with valid predictions...
Generated 9646 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Jamal Shead,Lauri Markkanen,Pascal Siakam,5.5,25.5,25.5,over,under,under,1,37.57,0.751,Med,Low,Low
1,Lauri Markkanen,Pascal Siakam,Oso Ighodaro,25.5,25.5,5.5,under,under,over,1,36.93,0.739,Low,Low,Med
2,Gradey Dick,Lauri Markkanen,Pascal Siakam,7.5,25.5,25.5,over,under,under,1,36.91,0.738,Low,Low,Low
3,Gradey Dick,Jamal Shead,Lauri Markkanen,7.5,5.5,25.5,over,over,under,1,36.07,0.721,Low,Med,Low
4,Jamal Shead,Lauri Markkanen,Oso Ighodaro,5.5,25.5,5.5,over,under,over,1,36.04,0.721,Med,Low,Med
